# Week 3 Homework - Multi-Provider LLM Pipeline, Evaluation & Hallucination Audit
**COM SCI-X 455 - Week 3**

Keys come from Colab Secrets only: `UCLA_OPENAI_API_KEY` (or `OPENAI_API_KEY`) and `ANTHROPIC_API_KEY`.
Optional `HF_TOKEN` adds Llama 3.1 8B through the Hugging Face Inference API.

All results are saved to `results/` (CSV + Markdown) so `analysis.md` can quote them.

In [1]:
!pip install -q -U openai anthropic pydantic pandas tabulate requests huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


In [2]:
import os
def get_secret(*names):
    """Read a key from Colab Secrets (or env vars). Keys are never printed."""
    try:
        from google.colab import userdata
        for n in names:
            try:
                v = userdata.get(n)
                if v:
                    return v
            except Exception:
                pass
    except ImportError:
        pass
    for n in names:
        if os.environ.get(n):
            return os.environ[n]
    return None

OPENAI_KEY = get_secret("UCLA_OPENAI_API_KEY", "OPENAI_API_KEY")
ANTHROPIC_KEY = get_secret("ANTHROPIC_API_KEY", "UCLA_ANTHROPIC_API_KEY", "CLAUDE_API_KEY")
if OPENAI_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_KEY
if ANTHROPIC_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_KEY
print("OpenAI key found:", bool(OPENAI_KEY), "| Anthropic key found:", bool(ANTHROPIC_KEY))

OpenAI key found: True | Anthropic key found: False


In [3]:
import time, json, random, re
from openai import OpenAI
import anthropic

# USD per 1M tokens (input, output). Official list prices; check the provider pages before reuse.
PRICES = {"gpt-4o": (2.50, 10.00), "gpt-4o-mini": (0.15, 0.60),
          "claude-sonnet-4-5": (3.00, 15.00), "llama-3.1-8b (HF)": (0.0, 0.0)}
GPT_MODEL = "gpt-4o"
CLAUDE_MODEL = "claude-sonnet-4-5"
LLAMA_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

oa = OpenAI() if OPENAI_KEY else None
an = anthropic.Anthropic() if ANTHROPIC_KEY else None
HF_TOKEN = get_secret("HF_TOKEN", "HUGGINGFACE_TOKEN")
hf = None
if HF_TOKEN:
    try:
        from huggingface_hub import InferenceClient
        hf = InferenceClient(model=LLAMA_MODEL, token=HF_TOKEN)
    except Exception as e:
        print("Llama (HF) not available:", e)

def with_retry(fn, tries=4):
    """Call fn(); on API errors wait 1s, 2s, 4s (+ jitter) and try again."""
    for i in range(tries):
        try:
            return fn()
        except Exception as e:
            if i == tries - 1:
                raise
            print(f"  retry {i+1}: {type(e).__name__}: {str(e)[:120]}")
            time.sleep(2 ** i + random.random())

def call(model, user, system=None, temperature=0, max_tokens=1200, json_mode=False):
    """One chat call to any provider. Returns text, latency, tokens and cost."""
    t0 = time.time()
    if model.startswith("gpt"):
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
        kw = dict(model=model, messages=msgs, temperature=temperature, max_tokens=max_tokens)
        if json_mode:
            kw["response_format"] = {"type": "json_object"}
        r = with_retry(lambda: oa.chat.completions.create(**kw))
        text, tin, tout = r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens
    elif model.startswith("claude"):
        kw = dict(model=model, max_tokens=max_tokens, temperature=temperature,
                  messages=[{"role": "user", "content": user}])
        if system:
            kw["system"] = system          # Anthropic: system is a top-level parameter
        r = with_retry(lambda: an.messages.create(**kw))
        text = "".join(b.text for b in r.content if b.type == "text")
        tin, tout = r.usage.input_tokens, r.usage.output_tokens
    else:  # Llama 3 via Hugging Face Inference API
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
        r = with_retry(lambda: hf.chat_completion(messages=msgs, max_tokens=max_tokens, temperature=max(temperature, 0.01)))
        text = r.choices[0].message.content
        tin = getattr(r.usage, "prompt_tokens", 0) if r.usage else 0
        tout = getattr(r.usage, "completion_tokens", 0) if r.usage else 0
    lat = time.time() - t0
    pin, pout = PRICES.get(model, (0, 0))
    return {"model": model, "text": text, "latency_s": round(lat, 2), "in_tok": tin, "out_tok": tout,
            "cost_usd": round(tin / 1e6 * pin + tout / 1e6 * pout, 6)}

MODELS = []
if oa: MODELS.append(GPT_MODEL)
if an: MODELS.append(CLAUDE_MODEL)
if hf: MODELS.append("llama-3.1-8b (HF)")
print("Models in this run:", MODELS)
if len(MODELS) < 2:
    print("WARNING: add ANTHROPIC_API_KEY (or HF_TOKEN) to Colab Secrets to compare two providers.")

def parse_json(text):
    """Pull the first JSON object out of a model reply (handles ```json fences)."""
    m = re.search(r"\{.*\}", text, re.S)
    return json.loads(m.group(0)) if m else None

import os, pandas as pd, requests
os.makedirs("results", exist_ok=True)
def save(df, name):
    df.to_csv(f"results/{name}.csv", index=False)
    open(f"results/{name}.md", "w").write(df.to_markdown(index=False))
    print(f"saved results/{name}.csv")

Models in this run: ['gpt-4o']


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


## Part 1 - Multi-Task LLM Pipeline

| Task | Input | Output check |
|---|---|---|
| A. Long-document summary | Wikipedia article "Head Start (program)" (first 1,000+ words) | exactly 3 sentences, faithful |
| B. Structured extraction | Wikipedia article "CalWORKs" (news-style policy text) | valid JSON with people, organisations, dates, key claims |
| C. Chain-of-thought | A family child care budget case with a known numeric answer | correct numbers, reasoning shown |

Texts are fetched from the public Wikipedia API at run time (no credentials).

In [4]:
def wiki_text(title, min_words=0, max_words=1400):
    r = requests.get("https://en.wikipedia.org/w/api.php", timeout=30,
                     headers={"User-Agent": "genai-course-homework/1.0 (student project)"},
                     params={"action": "query", "prop": "extracts", "explaintext": 1, "format": "json",
                             "redirects": 1, "titles": title})
    page = next(iter(r.json()["query"]["pages"].values()))
    words = page["extract"].split()
    assert len(words) >= min_words, f"{title}: only {len(words)} words"
    return " ".join(words[:max_words])

DOC_A = wiki_text("Head Start (program)", min_words=1000, max_words=1400)
DOC_B = wiki_text("CalWORKs", max_words=700)
print("Task A words:", len(DOC_A.split()), "| Task B words:", len(DOC_B.split()))

TASK_C = """A licensed family child care home has 2 infants, 6 toddlers and 3 school-age children.
Monthly tuition: infant $1,600, toddler $1,300, school-age (after school) $700.
Monthly costs: rent $2,500, assistant $3,200, and food at $4 per child per day for 22 days
(food is only for infants and toddlers).
Question 1: What is the monthly profit now?
Question 2: The provider can replace one toddler with one infant (same total count).
What is the new monthly profit, and how much does profit change?"""
TASK_C_ANSWER = {"profit_now": 6696, "profit_new": 6996, "change": 300}

TASKS = {
 "A_summary": dict(system="You are an editor for busy policy readers.",
                   user=f"Summarise the following article in exactly 3 sentences. Use only facts in the article.\n\n{DOC_A}"),
 "B_extraction": dict(system="You extract entities for a database. Output JSON only.",
                      user=f"""Extract entities from the article below. Return JSON with keys:
"people": [names], "organisations": [names], "dates": [dates or years as written],
"key_claims": [up to 5 short factual claims]. Only include items that appear in the text.

{DOC_B}""", json_mode=True),
 "C_reasoning": dict(system="You are a careful small-business advisor.",
                     user=TASK_C + "\n\nThink step by step and show every calculation. End with a line: FINAL: profit_now=..., profit_new=..., change=..."),
}

Task A words: 1400 | Task B words: 700


In [5]:
def auto_checks(task, text):
    """Rule-based checks that do not need a judge."""
    if task == "A_summary":
        n = len([s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s])
        return {"rule": f"{n} sentences", "rule_ok": n == 3}
    if task == "B_extraction":
        try:
            d = parse_json(text); ok = all(k in d for k in ("people", "organisations", "dates", "key_claims"))
            return {"rule": "valid JSON" if ok else "missing keys", "rule_ok": ok}
        except Exception:
            return {"rule": "invalid JSON", "rule_ok": False}
    nums = [int(x.replace(",", "")) for x in re.findall(r"\d[\d,]*", text.split("FINAL")[-1])]
    ok = all(v in nums for v in TASK_C_ANSWER.values())
    return {"rule": "numbers correct" if ok else f"numbers {nums[:3]}", "rule_ok": ok}

def judge_quality(task, source, answer):
    """1-5 rating by a GPT-4o judge with a fixed rubric (I review the ratings and notes)."""
    rubric = {"A_summary": "5 = exactly 3 sentences, covers the main points, no unsupported claims.",
              "B_extraction": "5 = valid JSON, entities complete and all present in the text, no invented items.",
              "C_reasoning": f"5 = correct final numbers {TASK_C_ANSWER}, clear steps."}[task]
    out = call(GPT_MODEL, f"""Rate the answer 1-5. Rubric: {rubric}
Source (may be truncated): {source[:6000]}
Answer: {answer}
Return JSON: {{"score": int, "reason": "one sentence"}}""", json_mode=True, max_tokens=200)
    return json.loads(out["text"])

part1 = []
for task, spec in TASKS.items():
    for m in MODELS:
        try:
            r = call(m, spec["user"], system=spec["system"], max_tokens=1500,
                     json_mode=spec.get("json_mode", False) and m.startswith("gpt"))
        except Exception as e:
            print(f"{task} / {m} failed after retries: {e}")
            continue
        chk = auto_checks(task, r["text"])
        j = judge_quality(task, spec["user"], r["text"])
        part1.append({"Task": task, "Model": m, "Latency (s)": r["latency_s"], "In tok": r["in_tok"],
                      "Out tok": r["out_tok"], "Cost ($)": r["cost_usd"], "Rule check": chk["rule"],
                      "Quality 1-5": j["score"], "Judge reason": j["reason"], "output": r["text"]})
        print(f"===== {task} | {m} | {r['latency_s']}s | ${r['cost_usd']} | {chk['rule']} | Q={j['score']}")
        print(r["text"][:1500], "\n")

===== A_summary | gpt-4o | 3.57s | $0.005685 | 4 sentences | Q=1
Head Start, established in 1965, is a comprehensive program by the U.S. Department of Health and Human Services aimed at providing early childhood education, health, nutrition, and parent involvement services to low-income families. Initially launched as a summer program, it has expanded over the years to serve over 1 million children annually across the U.S., including specialized services for migrant, seasonal, and homeless children. The program's governance includes a Policy Council composed of parents and community representatives, which plays a crucial role in decision-making and fundraising, while eligibility is primarily income-based, with provisions for children with disabilities and those from families experiencing emergencies. 

===== B_extraction | gpt-4o | 2.41s | $0.004765 | valid JSON | Q=5
{
    "people": [],
    "organisations": [
        "California Work Opportunities and Responsibility to Kids (CalWORKs)

In [6]:
p1 = pd.DataFrame(part1)
# Key observation: compare each model with the other provider on the same task
obs = []
for _, row in p1.iterrows():
    other = p1[(p1.Task == row.Task) & (p1.Model != row.Model)]
    if len(other):
        o = other.iloc[0]
        faster = "faster" if row["Latency (s)"] < o["Latency (s)"] else "slower"
        cheaper = "cheaper" if row["Cost ($)"] < o["Cost ($)"] else "more expensive"
        obs.append(f"{faster} and {cheaper} than {o.Model}; {len(row.output.split())} vs {len(o.output.split())} words; {row['Judge reason']}")
    else:
        obs.append(row["Judge reason"])
p1["Key observation"] = obs
table1 = p1[["Task", "Model", "Latency (s)", "Cost ($)", "Quality 1-5", "Rule check", "Key observation"]]
save(table1, "part1_results")
save(p1[["Task", "Model", "output"]], "part1_outputs")
table1

saved results/part1_results.csv
saved results/part1_outputs.csv


,Task,Model,Latency (s),Cost ($),Quality 1-5,Rule check,Key observation
0,A_summary,gpt-4o,3.57,0.005685,1,4 sentences,one sentence
1,B_extraction,gpt-4o,2.41,0.004765,5,valid JSON,"Valid JSON, entities complete and all present ..."
2,C_reasoning,gpt-4o,3.55,0.005528,5,numbers correct,The answer provides correct final numbers and ...


## Part 2 - Hallucination Audit

**Model audited:** GPT-4o at temperature=0.
**Domain:** history of AI and machine learning research (papers, authors, venues, numbers). I know this domain and
every ground-truth answer below comes from the original papers. Several questions ask for bibliographic details
(volume, pages, DOI) because those are where fabrication usually appears.

In [7]:
QA = [
 ("Who wrote the 1986 Nature paper that popularised backpropagation, and what was its title?", "David Rumelhart, Geoffrey Hinton, Ronald Williams; 'Learning representations by back-propagating errors'."),
 ("In which volume and on which pages of Nature was 'Learning representations by back-propagating errors' published?", "Nature volume 323, pages 533-536 (1986)."),
 ("What is the title, year and venue of the paper that introduced the Transformer?", "'Attention Is All You Need', 2017, NIPS/NeurIPS 2017 (arXiv 1706.03762)."),
 ("What BLEU score did the Transformer (big) report on WMT 2014 English-to-German?", "28.4 BLEU."),
 ("How many parameters did the Transformer 'big' model have in the original paper?", "About 213 million (Transformer base: about 65 million)."),
 ("What top-5 test error did AlexNet's winning ILSVRC-2012 entry achieve?", "15.3% top-5 error."),
 ("What is the DOI of the Communications of the ACM version of the AlexNet paper?", "10.1145/3065386 (CACM 2017). The original NIPS 2012 paper has no DOI."),
 ("How many layers did the deepest ResNet trained on ImageNet in the original ResNet paper have?", "152 layers."),
 ("What volume, issue and pages of Neural Computation published Hochreiter and Schmidhuber's LSTM paper?", "Volume 9, issue 8, pages 1735-1780 (1997)."),
 ("In which journal, volume and pages was the 2014 Dropout paper by Srivastava et al. published?", "Journal of Machine Learning Research, volume 15, pages 1929-1958 (2014)."),
 ("What default values does the Adam paper recommend for beta1, beta2 and epsilon?", "beta1 = 0.9, beta2 = 0.999, epsilon = 1e-8."),
 ("What is the title of the 2013 Mikolov et al. paper that introduced word2vec (CBOW and skip-gram)?", "'Efficient Estimation of Word Representations in Vector Space'."),
 ("How many parameters did the largest GPT-2 model have?", "1.5 billion (1,542 million)."),
 ("How many parameters do BERT-base and BERT-large have?", "BERT-base about 110 million, BERT-large about 340 million."),
 ("In what year and journal was Alan Turing's 'Computing Machinery and Intelligence' published, with volume and pages?", "1950, Mind, volume 59, issue 236, pages 433-460."),
 ("What was the title of Frank Rosenblatt's 1958 Psychological Review paper and its volume?", "'The perceptron: A probabilistic model for information storage and organization in the brain', Psychological Review volume 65 (issue 6), 1958."),
 ("Who received the 2018 ACM Turing Award and for what?", "Yoshua Bengio, Geoffrey Hinton and Yann LeCun, for conceptual and engineering breakthroughs that made deep neural networks a critical part of computing."),
 ("Who won the 2024 Nobel Prize in Physics?", "John Hopfield and Geoffrey Hinton."),
 ("What was the final score of the March 2016 AlphaGo vs Lee Sedol match?", "AlphaGo won 4-1."),
 ("Who wrote the paper that introduced latent diffusion models (Stable Diffusion), and where was it published?", "Rombach, Blattmann, Lorenz, Esser, Ommer; 'High-Resolution Image Synthesis with Latent Diffusion Models', CVPR 2022."),
]
assert len(QA) == 20
AUDIT_SYSTEM = "You are a research assistant. Answer precisely. On the last line write 'Confidence: N' where N is 0-100."
JUDGE_MODEL = CLAUDE_MODEL if an else GPT_MODEL   # use the other provider as grader when available

def grade(q, truth, answer):
    out = call(JUDGE_MODEL, f"""Question: {q}
Ground truth (from the original source): {truth}
Model answer: {answer}

Grade the answer against the ground truth.
correct = all stated facts match; partial = some match but something is missing or vague;
hallucinated = at least one stated fact is false or invented.
If hallucinated, choose a type: fabricated_citation (invented title/venue/volume/pages/DOI/authors),
wrong_statistic (wrong number), plausible_but_incorrect (wrong non-numeric fact),
confident_uncertainty_avoidance (vague or evasive answer stated confidently instead of admitting uncertainty).
Return JSON: {{"grade": "correct|partial|hallucinated", "type": "none|fabricated_citation|wrong_statistic|plausible_but_incorrect|confident_uncertainty_avoidance", "wrong_detail": "short"}}""",
              json_mode=JUDGE_MODEL.startswith("gpt"), max_tokens=300)
    return parse_json(out["text"])

audit = []
for i, (q, truth) in enumerate(QA, 1):
    a = call(GPT_MODEL, q, system=AUDIT_SYSTEM, temperature=0, max_tokens=400)["text"]
    conf = re.search(r"Confidence:\s*(\d+)", a)
    g = grade(q, truth, a)
    audit.append({"#": i, "Question": q, "Ground truth": truth, "Answer": a.strip(),
                  "Confidence": int(conf.group(1)) if conf else None,
                  "Grade": g["grade"], "Type": g["type"], "Wrong detail": g.get("wrong_detail", "")})
    print(f"Q{i:02d} [{g['grade']:12s}] conf={audit[-1]['Confidence']} {q[:70]}")
audit_df = pd.DataFrame(audit)
save(audit_df, "part2_audit")

Q01 [correct     ] conf=95 Who wrote the 1986 Nature paper that popularised backpropagation, and 
Q02 [correct     ] conf=95 In which volume and on which pages of Nature was 'Learning representat
Q03 [correct     ] conf=100 What is the title, year and venue of the paper that introduced the Tra
Q04 [correct     ] conf=95 What BLEU score did the Transformer (big) report on WMT 2014 English-t
Q05 [correct     ] conf=95 How many parameters did the Transformer 'big' model have in the origin
Q06 [correct     ] conf=100 What top-5 test error did AlexNet's winning ILSVRC-2012 entry achieve?
Q07 [correct     ] conf=95 What is the DOI of the Communications of the ACM version of the AlexNe
Q08 [correct     ] conf=100 How many layers did the deepest ResNet trained on ImageNet in the orig
Q09 [correct     ] conf=95 What volume, issue and pages of Neural Computation published Hochreite
Q10 [correct     ] conf=95 In which journal, volume and pages was the 2014 Dropout paper by Sriva
Q11 [correct     

In [8]:
counts = audit_df.Grade.value_counts()
X, Y, Z = counts.get("hallucinated", 0), counts.get("partial", 0), counts.get("correct", 0)
print(f"Hallucination rate: {X}/20 hallucinated, {Y}/20 partially correct, {Z}/20 correct")
tax = audit_df[audit_df.Grade == "hallucinated"].Type.value_counts().rename_axis("Type").reset_index(name="Count")
for t in ["fabricated_citation", "wrong_statistic", "plausible_but_incorrect", "confident_uncertainty_avoidance"]:
    if t not in set(tax.Type):
        tax.loc[len(tax)] = [t, 0]
save(tax, "part2_taxonomy")
print(tax.to_markdown(index=False))
audit_df[["#", "Grade", "Type", "Confidence", "Wrong detail"]]

Hallucination rate: 0/20 hallucinated, 3/20 partially correct, 17/20 correct
saved results/part2_taxonomy.csv
| Type                            |   Count |
|:--------------------------------|--------:|
| fabricated_citation             |       0 |
| wrong_statistic                 |       0 |
| plausible_but_incorrect         |       0 |
| confident_uncertainty_avoidance |       0 |


,#,Grade,Type,Confidence,Wrong detail
0,1,correct,none,95,
1,2,correct,none,95,
2,3,correct,none,100,
3,4,correct,none,95,
4,5,correct,none,95,
5,6,correct,none,100,
6,7,correct,none,95,
7,8,correct,none,100,
8,9,correct,none,95,
9,10,correct,none,95,


### Mitigation experiment
Take the 3 hallucinated answers with the highest stated confidence (if fewer than 3 hallucinated, fill with the
highest-confidence partial answers), add *"If you are not certain, say I don't know"* to the system prompt and re-run.

In [9]:
MITIGATED_SYSTEM = AUDIT_SYSTEM + " If you are not certain, say I don't know."
pool = audit_df[audit_df.Grade == "hallucinated"].sort_values("Confidence", ascending=False)
if len(pool) < 3:
    pool = pd.concat([pool, audit_df[audit_df.Grade == "partial"].sort_values("Confidence", ascending=False)])
pool = pool.head(3)
mit = []
for _, row in pool.iterrows():
    a2 = call(GPT_MODEL, row.Question, system=MITIGATED_SYSTEM, temperature=0, max_tokens=400)["text"]
    g2 = grade(row.Question, row["Ground truth"], a2)
    said_idk = "don't know" in a2.lower() or "do not know" in a2.lower() or "not certain" in a2.lower()
    mit.append({"#": row["#"], "Before": f"{row.Grade} ({row.Type}), conf {row.Confidence}",
                "After": f"{g2['grade']} ({g2['type']})", "Said I don't know": said_idk,
                "After answer": a2.strip()[:250]})
mit_df = pd.DataFrame(mit)
save(mit_df, "part2_mitigation")
mit_df

saved results/part2_mitigation.csv


,#,Before,After,Said I don't know,After answer
0,13,"partial (wrong_statistic), conf 100",partial (wrong_statistic),False,The largest GPT-2 model has 1.5 billion parame...
1,15,"partial (none), conf 100",partial (none),False,Alan Turing's paper 'Computing Machinery and I...
2,20,"partial (none), conf 95",partial (plausible_but_incorrect),False,The paper that introduced latent diffusion mod...


## Part 3 - Role-Specific Prompt Comparison

**Recurring task (from my work at USCIC):** after each weekly staff meeting about the parent-intake program, turn the
meeting transcript into a short summary with decisions and action items (owner, task, due date) that I paste into
our team chat. The same transcript is used for all 4 versions.

In [10]:
TRANSCRIPT = """Weekly intake meeting - Monday
Lin: We got 23 new parent forms last week. 9 were Spanish, 8 Chinese, 6 English.
Maria: The Spanish form still has the old phone field. Parents type the number twice. I can fix it by Wednesday.
Lin: Good. Next, the ZIP code match. Three families in 91770 had no daycare with open seats.
David: I can call the four licensed homes in 91770 and 91776 this week to ask about infant openings.
Lin: Please do that by Friday. Also, we agreed last month to keep parent contact data encrypted. Is that done?
Maria: Encryption is on for the database, but the CSV export still shows phone numbers in plain text.
Lin: Then we stop CSV exports until that is fixed. David, can you remove the export button by Thursday?
David: Yes.
Lin: Last thing: the board wants a one-page update with numbers. I will write it and send it next Monday.
Maria: Should we add Vietnamese to the form?
Lin: Not yet. We decide after we see October numbers."""

EXPECTED = {"decisions": ["stop CSV exports until phone numbers are protected", "Vietnamese form decided after October numbers"],
            "actions": [("Maria", "fix Spanish phone field", "Wednesday"), ("David", "call 4 licensed homes in 91770/91776", "Friday"),
                        ("David", "remove CSV export button", "Thursday"), ("Lin", "write one-page board update", "next Monday")]}
FORMAT_SPEC = "Format: a 2-sentence summary, then 'Decisions:' as a bullet list, then 'Action items:' as bullets in the form 'Owner - task - due date'."

EX1_IN = """Ana: The flyer printer is broken. I'll ask the vendor for a quote by Tuesday.
Ben: We decided to move the parent workshop to Saturday."""
EX1_OUT = """Summary: The flyer printer needs repair. The parent workshop moves to Saturday.
Decisions:
- Parent workshop moves to Saturday
Action items:
- Ana - get printer repair quote - Tuesday"""
EX2_IN = """Kim: 12 providers joined the site in September. I'll update the provider list by the 5th.
Joe: Let's keep the newsletter monthly. I'll draft October's by Friday."""
EX2_OUT = """Summary: 12 providers joined in September. The newsletter stays monthly.
Decisions:
- Newsletter stays monthly
Action items:
- Kim - update provider list - the 5th
- Joe - draft October newsletter - Friday"""

PROMPTS = {
 "V1 zero-shot": dict(system=None, user=f"Summarise this meeting transcript with decisions and action items.\n{FORMAT_SPEC}\n\n{TRANSCRIPT}"),
 "V2 few-shot": dict(system=None, user=f"""Summarise meeting transcripts. {FORMAT_SPEC}

Example 1 input:
{EX1_IN}
Example 1 output:
{EX1_OUT}

Example 2 input:
{EX2_IN}
Example 2 output:
{EX2_OUT}

Now the real input:
{TRANSCRIPT}
Output:"""),
 "V3 chain-of-thought": dict(system=None, user=f"""Summarise this meeting transcript. {FORMAT_SPEC}
Before the final output, think step by step: (1) list every speaker turn that contains a commitment,
(2) decide which statements are decisions vs. open questions, (3) find the owner and due date for each commitment.
Show this reasoning under 'Reasoning:', then give the final output under 'Final:'.

{TRANSCRIPT}"""),
}

In [11]:
from pydantic import BaseModel, Field, ValidationError
class ActionItem(BaseModel):
    owner: str
    task: str
    due: str = Field(description="due date as said in the meeting, or 'not set'")
class MeetingSummary(BaseModel):
    summary: str = Field(description="2 sentences")
    decisions: list[str]
    action_items: list[ActionItem]
    open_questions: list[str] = []

ROLE_SYSTEM = ("You are the program coordinator's assistant at a nonprofit that helps multilingual families find child care. "
               "You turn meeting transcripts into records for the team chat. Only record what was said; never invent owners or dates. "
               "Return JSON that matches this schema:\n" + json.dumps(MeetingSummary.model_json_schema()))
PROMPTS["V4 role + Pydantic"] = dict(system=ROLE_SYSTEM, user=TRANSCRIPT, json_mode=True)

def to_text(ms: MeetingSummary):
    return ("Summary: " + ms.summary + "\nDecisions:\n" + "\n".join("- " + d for d in ms.decisions) +
            "\nAction items:\n" + "\n".join(f"- {a.owner} - {a.task} - {a.due}" for a in ms.action_items))

P3_MODEL = GPT_MODEL
p3_out = {}
for name, spec in PROMPTS.items():
    r = call(P3_MODEL, spec["user"], system=spec["system"], max_tokens=1200, json_mode=spec.get("json_mode", False))
    text = r["text"]
    if name.startswith("V4"):
        try:
            text = to_text(MeetingSummary.model_validate(parse_json(r["text"])))
            text += "\n[Pydantic validation: passed]"
        except (ValidationError, TypeError) as e:
            text += f"\n[Pydantic validation: FAILED {e}]"
    p3_out[name] = {"text": text, "tokens": r["in_tok"] + r["out_tok"], "cost": r["cost_usd"]}
    print(f"================ {name} ({r['in_tok']}+{r['out_tok']} tokens)\n{text}\n")

================ V1 zero-shot (282+165 tokens)
The meeting focused on addressing issues with parent forms and ensuring data security, while also planning for future updates and outreach. Key decisions included fixing the Spanish form, halting CSV exports until encryption issues are resolved, and delaying the addition of Vietnamese to the form until further data is reviewed.

Decisions:
- Fix the Spanish form's phone field by Wednesday.
- Stop CSV exports until phone number encryption is fixed.
- Delay adding Vietnamese to the form until October numbers are reviewed.

Action items:
- Maria - Fix the Spanish form's phone field - Due by Wednesday.
- David - Call licensed homes in 91770 and 91776 about infant openings - Due by Friday.
- David - Remove the CSV export button - Due by Thursday.
- Lin - Write and send a one-page update with numbers - Due by next Monday.

================ V2 few-shot (454+125 tokens)
Summary: 23 new parent forms were received last week, with issues noted in the

In [12]:
def score_version(name, text):
    final = text.split("Final:")[-1]
    # Format: rule check of the requested layout
    has_dec = "Decisions:" in final; has_act = "Action items:" in final
    act_lines = [l for l in final.splitlines() if l.strip().startswith("-") and l.count(" - ") >= 2]
    fmt = 1 + 2 * has_dec + 1 * has_act + (1 if len(act_lines) >= 3 else 0)
    if "Reasoning:" in text:
        fmt = max(1, fmt - 1)    # extra reasoning text has to be cut before pasting into chat
    # Correctness + usefulness: judge against my expected list
    j = json.loads(call(GPT_MODEL, f"""Expected decisions: {EXPECTED['decisions']}
Expected action items (owner, task, due): {EXPECTED['actions']}
Candidate output:
{final}

Score 1-5 each. correctness = how many expected decisions/actions are present with right owner and due date,
and nothing invented. usefulness = could a team member paste this into chat without edits?
Return JSON: {{"correctness": int, "usefulness": int, "notes": "one sentence naming the main problem"}}""",
                       json_mode=True, max_tokens=200)["text"])
    return {"Version": name, "Correctness": j["correctness"], "Format": min(fmt, 5),
            "Usefulness": j["usefulness"], "Tokens": p3_out[name]["tokens"], "Notes": j["notes"]}

p3 = pd.DataFrame([score_version(n, v["text"]) for n, v in p3_out.items()])
save(p3, "part3_eval")
save(pd.DataFrame([{"Version": k, "Output": v["text"]} for k, v in p3_out.items()]), "part3_outputs")
p3

saved results/part3_eval.csv
saved results/part3_outputs.csv


,Version,Correctness,Format,Usefulness,Tokens,Notes
0,V1 zero-shot,5,5,5,447,The output accurately reflects all expected de...
1,V2 few-shot,4,5,4,579,The summary incorrectly states that encryption...
2,V3 chain-of-thought,4,4,4,717,The action item for David about calling licens...
3,V4 role + Pydantic,5,5,5,714,The output accurately reflects all expected de...


Written analyses (Part 1 model choice, Part 2 hallucination pattern, Part 3 recommendation memo) are in
`analysis.md`. All tables are in `results/`.

In [13]:
# Download results to add them to the repo (week3/homework/results/)
import shutil
shutil.make_archive("week3_results", "zip", "results")
try:
    from google.colab import files; files.download("week3_results.zip")
except Exception:
    print("results zipped: week3_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>